# QuadNet Correction — Backward-Facing Step

Self-contained notebook that trains a **QuadNet** spatially-dependent
quadratic correction for the backward-facing step ROM and visualises the
results.

**Sections:**
1. Imports and setup
2. Model training
3. Evaluation (corrected vs baseline POD-RBF)
4. Comparison plots
5. Correction operator analysis
6. Gradient analysis of exact corrections

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

from pina import LabelTensor, Trainer
from pina.callbacks import MetricTracker
from pina.loss import LpLoss
from pytorch_lightning.callbacks import EarlyStopping

from problems.setup_backstep import BackstepProblem
from nns.quadnet import QuadNet
from rom.corrected_rom import CorrectedROM
from rom.pod_rbf import err, PODRBF
from utils.plotting import plot
torch.manual_seed(42)

os.makedirs("img", exist_ok=True)
print("Imports OK")

## 1. Setup

In [ ]:
# --- Parameters (reduce epochs/train_size for quick testing) ---
FIELD      = "mag(v)"
REDDIM     = 3
TRAIN_SIZE = 400
EPOCHS     = 500

# --- Load data and build POD + RBF ---
backstep = BackstepProblem(FIELD, REDDIM, subset=None,
                           train_size=TRAIN_SIZE, device='cpu')
data      = backstep.data
pod       = backstep.pod
rbf       = backstep.rbf
params_train     = backstep.params_train
params_test      = backstep.params_test
snapshots_train  = backstep.snapshots_train
snapshots_test   = backstep.snapshots_test
problem  = backstep.problem

print(f"Ndof={backstep.Ndof}  reddim={REDDIM}  train={TRAIN_SIZE}")

In [ ]:
# --- Build the QuadNet correction network and CorrectedROM ---
corr_net = QuadNet(backstep.modes, backstep.coords, scaler=backstep.scaler)

rom = CorrectedROM(
    problem=problem,
    reduction_network=pod,
    interpolation_network=rbf,
    correction_network=corr_net,
    loss=torch.nn.MSELoss(),
)
print(f"ROM created: {sum(p.numel() for p in rom.parameters())} trainable params")

## 2. Training

In [ ]:
trainer = Trainer(
    solver=rom,
    max_epochs=EPOCHS,
    accelerator="cpu",
    callbacks=[
        MetricTracker(),
        EarlyStopping(monitor="loss_corr", patience=500,
                      stopping_threshold=1e-4, check_on_train_epoch_end=True),
    ],
    batch_size=TRAIN_SIZE,
)
trainer.train()
rom.eval()
print("Training complete")

## 3. Evaluation

In [ ]:
# --- Corrected ROM errors ---
predicted_train = rom(params_train)
predicted_test  = rom(params_test)

def rel_error(true, pred):
    return (torch.linalg.norm(true - pred, dim=-1)
            / torch.linalg.norm(true, dim=-1)).tensor.cpu().detach().numpy()

corr_train_err = rel_error(snapshots_train, predicted_train)
corr_test_err  = rel_error(snapshots_test,  predicted_test)
print(f"Corrected ROM  --  train: {corr_train_err.mean():.6f} +/- {corr_train_err.std():.6f}"
      f"   test: {corr_test_err.mean():.6f} +/- {corr_test_err.std():.6f}")

In [ ]:
# --- Baseline POD-RBF errors ---
pod_rbf = PODRBF(pod_rank=REDDIM, rbf_kernel="linear")
pod_rbf.fit(params_train, snapshots_train)

pod_train_pred = pod_rbf(params_train)
pod_test_pred  = pod_rbf(params_test)

pod_train_err = rel_error(snapshots_train, pod_train_pred)
pod_test_err  = rel_error(snapshots_test,  pod_test_pred)
print(f"POD-RBF        --  train: {pod_train_err.mean():.6f} +/- {pod_train_err.std():.6f}"
      f"   test: {pod_test_err.mean():.6f} +/- {pod_test_err.std():.6f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot([corr_train_err, corr_test_err, pod_train_err, pod_test_err],
           tick_labels=["Corr\ntrain", "Corr\ntest", "POD\ntrain", "POD\ntest"],
           patch_artist=True,
           boxprops=dict(facecolor="lightblue"),
           medianprops=dict(color="k"))
ax.set_yscale("log")
ax.set_ylabel("Relative error")
ax.set_title("Corrected ROM vs baseline POD-RBF")
fig.tight_layout()
plt.show()

## 4. Spatial comparison -- single test snapshot

In [ ]:
ind = 2
snap     = snapshots_test[ind].tensor.cpu().detach().numpy().ravel()
pred     = predicted_test[ind].tensor.cpu().detach().numpy().ravel()
pod_pred = pod_test_pred[ind].tensor.cpu().detach().numpy().ravel()

fields = [snap, pred, pod_pred, snap - pred, snap - pod_pred]
labels = ["Truth", "Corrected ROM", "POD-RBF",
          "Error corrected", "Error POD"]
plot(data.triang, fields, labels, filename="img/quadnet_mu_backstep_compare.png")

## 5. Correction: approximate vs exact

In [ ]:
exact_corr  = CorrectedROM.compute_exact_correction(pod, snapshots_test)
approx_corr = corr_net(params_test, rbf(params_test))
corr_scaler = rom.neural_net['correction_network'].scaler
if corr_scaler is not None:
    approx_corr = corr_scaler.inverse_transform(approx_corr)

approx = approx_corr[ind].cpu().detach().numpy().ravel()
exact  = exact_corr[ind].tensor.cpu().detach().numpy().ravel()

fields = [approx, exact, approx - exact]
labels = ["Approx correction", "Exact correction", "Error"]
plot(data.triang, fields, labels, filename="img/quadnet_mu_backstep_correction.png")

## 6. Operator C -- entry maps at different mu values

In [ ]:
mus = torch.tensor([[0.2], [0.4], [0.6], [0.8]])
K = REDDIM * (REDDIM + 1) // 2

for i, mu_val in enumerate(mus):
    mu_batch = LabelTensor(mu_val.unsqueeze(0), ["mu"])
    c = corr_net.C(mu_batch).tensor.cpu().detach().numpy()  # (N_dof, K)
    fields_c = [c[:, j] for j in range(K)]
    labels_c = [f"c{j}" for j in range(K)]
    plot(data.triang, fields_c, labels_c,
         filename=f"img/quadnet_mu_backstep_C_mu{mu_val.item():.1f}.png")


## 7. Operator C -- density of entry values

In [ ]:
c_all = corr_net.C(params_test).tensor.cpu().detach().numpy()  # (N_dof, K)

fig, axs = plt.subplots(1, K, figsize=(3 * K, 3))
x_range = np.linspace(c_all.min(), c_all.max(), 200)
for j, ax in enumerate(axs):
    dens = gaussian_kde(c_all[:, j])
    ax.fill_between(x_range, 0, dens(x_range), alpha=0.6, color="grey")
    ax.plot(x_range, dens(x_range), "k")
    ax.set_title(f"C[:, {j}]")
    ax.set_xlabel("value")
axs[0].set_ylabel("density")
fig.suptitle("Distribution of correction-operator entries", y=1.02)
fig.tight_layout()
plt.show()

## 8. Per-sample relative error distribution

In [ ]:
per_sample_err = rel_error(snapshots_test, predicted_test)

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(per_sample_err, bins=20, color="grey", edgecolor="k", alpha=0.7)
ax.axvline(per_sample_err.mean(), color="r", ls="--",
           label=f"mean={per_sample_err.mean():.4f}")
ax.set_xlabel("Relative error")
ax.set_ylabel("Count")
ax.set_title("Test-set relative error distribution")
ax.legend()
fig.tight_layout()
plt.show()

## 9. Gradient analysis of exact corrections

In [ ]:
def compute_node_gradients(triangulation, values):
    """Area-weighted average gradient magnitude at each mesh node."""
    x, y = triangulation.x, triangulation.y
    tris  = triangulation.triangles
    grads = np.zeros_like(x)
    areas = np.zeros_like(x)
    for tri in tris:
        p = np.column_stack([x[tri], y[tri]])
        mat = p[1:] - p[0]
        area = 0.5 * abs(np.linalg.det(mat))
        dv = values[tri[1:]] - values[0]
        g  = np.linalg.solve(mat.T, dv)
        for k in range(3):
            grads[tri[k]] += np.linalg.norm(g) * area
            areas[tri[k]] += area
    grads /= np.where(areas > 0, areas, 1.0)
    return grads

triang = data.triang
corrections = backstep.exact_correction.tensor.cpu().detach().numpy()
avg_grad = np.zeros(corrections.shape[1])
for i in range(corrections.shape[0]):
    avg_grad += compute_node_gradients(triang, corrections[i])
avg_grad /= corrections.shape[0]

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
tp0 = ax[0].tricontourf(triang, corrections[-1], levels=20)
plt.colorbar(tp0, ax=ax[0])
ax[0].set_title("Exact correction (last sample)")
tp1 = ax[1].tricontourf(triang, avg_grad, levels=20)
plt.colorbar(tp1, ax=ax[1])
ax[1].set_title("Avg. gradient magnitude over all corrections")
fig.tight_layout()
plt.show()

## 10. Probability-density based spatial sampling

In [ ]:
N = backstep.Ndof
p = np.exp(-np.mean(np.abs(corrections), axis=0)
           / (np.mean(np.abs(corrections), axis=0).mean() + 1e-6))
p /= p.sum()

fig, ax = plt.subplots(figsize=(7, 5))
tp = ax.tricontourf(triang, p, levels=20)
plt.colorbar(tp, ax=ax)
ax.set_title("Probability density for spatial sampling")
fig.tight_layout()
plt.show()

frac = 0.2
ind_importance = np.random.choice(N, int(frac * N), replace=False, p=p)
ind_uniform    = np.random.choice(N, int(frac * N), replace=False)

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for a, idx, title in [(ax[0], ind_importance, 'Importance-sampled'),
                       (ax[1], ind_uniform,    'Uniform random')]:
    a.tricontourf(triang, np.zeros(N), levels=1, colors=['white'])
    a.triplot(triang, c='grey', alpha=0.3, lw=0.3)
    a.scatter(triang.x[idx], triang.y[idx], c='k', s=1)
    a.set_title(title)
    a.set_aspect('equal')
fig.tight_layout()
plt.show()

### Done

All sections executed successfully.  To improve accuracy, increase
`EPOCHS` and/or `TRAIN_SIZE` in the setup cell above.